In [67]:
import pandas as pd

In [68]:
import pandas as pd
from IPython.display import display

# Run once per kernel session: normal DataFrame displays now use dark colors.
# Keep pandas' normal row/column truncation and number formatting.
pd.set_option("display.notebook_repr_html", True)
_dataframe_dark_css = """
<style>
.dark-dataframe {
    background-color: #1e1e1e !important;
    color: #e0e0e0 !important;
    padding: 8px;
    overflow-x: auto;
}
.dark-dataframe table.dataframe,
.dark-dataframe table.dataframe td,
.dark-dataframe table.dataframe th {
    background-color: #1e1e1e !important;
    color: #e0e0e0 !important;
    border-color: #444444 !important;
}
.dark-dataframe table.dataframe th {
    background-color: #292929 !important;
}
.dark-dataframe table.dataframe tbody tr:hover td,
.dark-dataframe table.dataframe tbody tr:hover th {
    background-color: #333333 !important;
}
</style>
"""
_ = get_ipython().display_formatter.formatters["text/html"].for_type(
    pd.DataFrame,
    lambda frame: (
        _dataframe_dark_css
        + '<div class="dark-dataframe">'
        + frame._repr_html_()
        + '</div>'
    ),
)


<div dir="rtl" style="direction: rtl; text-align: right; line-height: 1.9;">

## ۱. اعتبارسنجی فایل <bdi dir="ltr">CSV</bdi> و قرنطینه رکوردهای نامعتبر در <bdi dir="ltr">Data Factory</bdi>

<p dir="rtl" style="direction: rtl; text-align: right;">در این بخش، فایل ورودی مرحله سوم با موتور عمومی پردازش <bdi dir="ltr">CSV</bdi> در پکیج مشترک بررسی می‌شود. قواعد اعتبارسنجی در یک «قرارداد داده» مستقل تعریف شده‌اند. این قرارداد، نوع ستون‌ها، نحوه برخورد با مقادیر گمشده، دامنه مقادیر مجاز، شرایط هشدار و کلید یکتا را مشخص می‌کند. موتور پردازش به قواعد اختصاصی مجموعه‌داده <bdi dir="ltr">NYC</bdi> یا نام ستون‌های داده‌های اعتباری وابسته نیست.</p>

<p dir="rtl" style="direction: rtl; text-align: right;">فایل خام در طول پردازش تغییر نمی‌کند. برای هر اجرا، رکوردهای پذیرفته‌شده، رکوردهای قرنطینه‌شده همراه با همه دلایل رد شدن، گزارش کیفیت و نسخه کامل قرارداد در پوشه‌ای مستقل ذخیره می‌شوند.</p>

<p dir="rtl" style="direction: rtl; text-align: right;">در قرارداد اولیه، مقادیر گمشده درآمد و تعداد افراد تحت تکفل با ثبت هشدار حفظ می‌شوند. سن صفر، خطای قطعی محسوب می‌شود و باعث قرنطینه رکورد می‌شود. نسبت استفاده از اعتبار بیش از یک و مقادیر ۹۶ و ۹۸ در ستون‌های دیرکرد نیز هشدار ایجاد می‌کنند؛ این مقادیر به‌صورت خودکار حذف یا اصلاح نمی‌شوند.</p>

<p dir="rtl" style="direction: rtl; text-align: right;">این بخش به آماده‌سازی داده و کنترل کیفیت اختصاص دارد. در بخش‌های بعد، جدول پایگاه داده ساخته می‌شود و محتوای فایل پذیرفته‌شده از طریق <bdi dir="ltr">SQLAlchemy</bdi> در آن بارگذاری می‌شود. جایگزینی مقادیر گمشده و مقیاس‌بندی باید پس از تقسیم داده‌ها انجام شوند؛ پارامترهای این تبدیل‌ها باید فقط از داده‌های آموزش (<bdi dir="ltr">Train</bdi>) به دست آیند.</p>

</div>

In [69]:
from pathlib import Path
from itertools import islice
import hashlib
import json
import pandas as pd

from data_factory.csv_contract import load_contract

current = Path.cwd().resolve()
candidates = [current, *current.parents]
candidates += [p / "machine_learning_challenge" for p in candidates[:]]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "config/credit_validation.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the project or its stage directory.")
CONFIG_PATH = PROJECT_ROOT / "config/credit_validation.json"
csv_settings = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
RAW_PATH = (CONFIG_PATH.parent / csv_settings["source"]["path"]).resolve()
OUTPUT_ROOT = (CONFIG_PATH.parent / csv_settings["output_dir"]).resolve()

def file_sha256(path):
    with Path(path).open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()

raw_hash_before = file_sha256(RAW_PATH)
previous_pointer = OUTPUT_ROOT / "latest.json"
previous_report = None
if previous_pointer.exists():
    previous = json.loads(previous_pointer.read_text(encoding="utf-8"))
    previous_report = json.loads((OUTPUT_ROOT / previous["report"]).read_text(encoding="utf-8"))
print("Project:", PROJECT_ROOT.name)
print("Input:", RAW_PATH.relative_to(PROJECT_ROOT))

Project: machine_learning_challenge
Input: data\stage_03\raw\cs-training.csv


<div dir="rtl" style="direction: rtl; text-align: right; line-height: 1.9;">

### قرارداد اولیه داده

<p dir="rtl" style="direction: rtl; text-align: right;">گزینه <code dir="ltr" style="unicode-bidi: isolate;">nullable</code> مشخص می‌کند که مقدار گمشده در یک ستون پذیرفته می‌شود یا خیر. گزینه <code dir="ltr" style="unicode-bidi: isolate;">on_invalid</code> نیز نحوه برخورد با مقادیر نامعتبر را تعیین می‌کند.</p>

<p dir="rtl" style="direction: rtl; text-align: right;">قواعد <code dir="ltr" style="unicode-bidi: isolate;">min</code> و <code dir="ltr" style="unicode-bidi: isolate;">allowed</code> الزام‌آورند؛ نقض آن‌ها خطای اعتبارسنجی ایجاد می‌کند. در مقابل، قواعد <code dir="ltr" style="unicode-bidi: isolate;">warn_values</code> و <code dir="ltr" style="unicode-bidi: isolate;">warn_max</code> فقط هشدار ثبت می‌کنند.</p>

<p dir="rtl" style="direction: rtl; text-align: right;">ستون‌های دارای نقش شناسه (<code dir="ltr" style="unicode-bidi: isolate;">identifier</code>) یا متغیر هدف (<code dir="ltr" style="unicode-bidi: isolate;">target</code>) از ویژگی‌های ورودی مدل جدا نگه داشته می‌شوند. جدول زیر مستقیماً از قرارداد داده ساخته می‌شود.</p>

</div>

In [70]:
credit_contract = load_contract(CONFIG_PATH.parent / csv_settings["contract"])
contract_table = pd.DataFrame(credit_contract["fields"])
display(contract_table.reindex(columns=[
    "source", "name", "type", "role", "nullable", "required_presence", "on_invalid",
    "min", "max", "allowed", "warn_values", "warn_max", "description",
]).fillna(""))
print("Primary key:", credit_contract["primary_key"])
print("Maximum quarantine fraction:", credit_contract["max_quarantine_fraction"])

,source,name,type,role,nullable,required_presence,on_invalid,min,max,allowed,warn_values,warn_max,description
0,,row_id,integer,identifier,False,True,quarantine,1.0,,,,,Original unnamed CSV row identifier; provenanc...
1,SeriousDlqin2yrs,SeriousDlqin2yrs,integer,target,False,True,quarantine,,,"[0, 1]",,,Binary outcome; missing or nonbinary labels qu...
2,RevolvingUtilizationOfUnsecuredLines,RevolvingUtilizationOfUnsecuredLines,number,feature,False,True,quarantine,0.0,,,,1.0,Nonnegative utilization ratio. Values above on...
3,age,age,integer,feature,False,True,quarantine,1.0,,,,,Positive recorded age; zero is an explicit har...
4,NumberOfTime30-59DaysPastDueNotWorse,NumberOfTime30-59DaysPastDueNotWorse,integer,feature,False,True,quarantine,0.0,,,"[96, 98]",,Nonnegative integer count. Observed 96/98 valu...
5,DebtRatio,DebtRatio,number,feature,False,True,quarantine,0.0,,,,,Nonnegative ratio with no arbitrary maximum; v...
6,MonthlyIncome,MonthlyIncome,number,feature,True,True,quarantine,0.0,,,,,Missing income remains null with a warning. In...
7,NumberOfOpenCreditLinesAndLoans,NumberOfOpenCreditLinesAndLoans,integer,feature,False,True,quarantine,0.0,,,,,Nonnegative integer count.
8,NumberOfTimes90DaysLate,NumberOfTimes90DaysLate,integer,feature,False,True,quarantine,0.0,,,"[96, 98]",,Nonnegative integer count; 96/98 are retained ...
9,NumberRealEstateLoansOrLines,NumberRealEstateLoansOrLines,integer,feature,False,True,quarantine,0.0,,,,,Nonnegative integer count.


Primary key: ['row_id']
Maximum quarantine fraction: 0.01


<div dir="rtl" style="direction: rtl; text-align: right; line-height: 1.9;">

### اجرای اعتبارسنجی

<p dir="rtl" style="direction: rtl; text-align: right;">کد سلول بعد، اعتبارسنجی را اجرا می‌کند و معادل دستور زیر در ترمینال است:</p>

<div dir="ltr" style="text-align: left; unicode-bidi: isolate;">

```powershell
python -m data_factory validate-csv --config config/credit_validation.json
```

</div>

<p dir="rtl" style="direction: rtl; text-align: right;">هر رکورد دقیقاً در یکی از دو گروه «پذیرفته‌شده» یا «قرنطینه‌شده» قرار می‌گیرد. اگر نسبت رکوردهای قرنطینه‌شده به کل رکوردها از آستانه مجاز بیشتر باشد، ابتدا گزارش و رکوردها ذخیره می‌شوند و سپس خطا اعلام می‌شود. برابر بودن این نسبت با آستانه مجاز، مانع پذیرش اجرا نمی‌شود.</p>

<p dir="rtl" style="direction: rtl; text-align: right;">اگر ساختار فایل نامعتبر باشد یا فایل خام هنگام پردازش تغییر کند، خروجی جدیدی منتشر نمی‌شود.</p>


در ادامه هم هر فایل csv که اپلود کردیم تنها کافی است فایل های کانفیگ inventory_validation.json و همینطور inventory_schema.json  رو بسته به اون فایل بسازیم و کد های data factory را روی ان ران کینم :)
</div>

In [71]:
import importlib
import data_factory.csv_quality as csv_quality

# Refresh the module when this cell is rerun in an existing kernel.
csv_quality = importlib.reload(csv_quality)
validation_result = csv_quality.validate_csv(CONFIG_PATH)
RUN_DIR = Path(validation_result["output_dir"])
display(pd.DataFrame([{
    key: validation_result[key] for key in [
        "dataset", "status", "raw_records", "accepted_records", "quarantined_records",
        "quarantine_fraction", "max_quarantine_fraction",
    ]
}]))
print("Run directory:", RUN_DIR.relative_to(PROJECT_ROOT))
print("Accepted CSV:", (RUN_DIR / "accepted.csv").relative_to(PROJECT_ROOT))

Validated 25,000 CSV records: accepted=25,000, quarantined=0
Validated 50,000 CSV records: accepted=50,000, quarantined=0
Validated 75,000 CSV records: accepted=74,999, quarantined=1
Validated 100,000 CSV records: accepted=99,999, quarantined=1
Validated 125,000 CSV records: accepted=124,999, quarantined=1
Validated 150,000 CSV records: accepted=149,999, quarantined=1


,dataset,status,raw_records,accepted_records,quarantined_records,quarantine_fraction,max_quarantine_fraction
0,Give Me Some Credit,passed,150000,149999,1,0.000007,0.01


Run directory: data\stage_03\curated\validation\daa6d8270f11093efe1e54c1614c3a3e348fffc2a4f876f533ad377289265259
Accepted CSV: data\stage_03\curated\validation\daa6d8270f11093efe1e54c1614c3a3e348fffc2a4f876f533ad377289265259\accepted.csv


<div dir="rtl" style="direction: rtl; text-align: right; line-height: 1.9;">

### تفسیر گزارش کیفیت و قرنطینه

<p dir="rtl" style="direction: rtl; text-align: right;">آمار کیفیت زیر فقط بر اساس رکوردهای پذیرفته‌شده محاسبه می‌شود؛ رکوردهای قرنطینه‌شده در این آمار لحاظ نمی‌شوند. مقدار گمشده (<code dir="ltr" style="unicode-bidi: isolate;">null</code>) با صفر تفاوت دارد و در این مرحله هیچ مقداری با میانگین یا میانه جایگزین نمی‌شود.</p>

<p dir="rtl" style="direction: rtl; text-align: right;">برای هر کد هشدار، تعداد رکوردهای پذیرفته‌شده‌ای که آن هشدار را دارند نمایش داده می‌شود. یک رکورد ممکن است چند هشدار داشته باشد؛ بنابراین، تعدادهای مربوط به هشدارهای مختلف لزوماً مستقل از یکدیگر نیستند.</p>

<p dir="rtl" style="direction: rtl; text-align: right;">در نمونه رکوردهای قرنطینه‌شده، نام فایل ورودی، شماره رکورد و دلایل خطا نمایش داده می‌شوند.</p>

</div>

In [72]:
profile = json.loads((RUN_DIR / "quality_profile.json").read_text(encoding="utf-8"))
display(pd.DataFrame([
    {"column": name, "missing_count": item["missing_count"],
     "missing_percent": item["missing_percent"], "unique_non_missing": item["unique_non_missing"]}
    for name, item in profile["columns"].items()
]))
display(pd.DataFrame([
    {"warning": warning, "accepted_records": count}
    for warning, count in validation_result["warning_counts"].items()
]))
with (RUN_DIR / "quarantine.jsonl").open(encoding="utf-8") as file:
    quarantine_sample = [json.loads(line) for line in islice(file, 5)]
display(pd.DataFrame([
    {"source_file": row["source_file"], "source_row": row["source_row"],
     "reason_codes": row["reason_codes"], "errors": row["errors"]}
    for row in quarantine_sample
]))
print("Accepted target counts:", validation_result["target_counts"])
print("Model features:", validation_result["feature_columns"])

,column,missing_count,missing_percent,unique_non_missing
0,DebtRatio,0,0.000000,114193
1,MonthlyIncome,29731,19.820799,13594
2,NumberOfDependents,3924,2.616017,13
3,NumberOfOpenCreditLinesAndLoans,0,0.000000,58
4,NumberOfTime30-59DaysPastDueNotWorse,0,0.000000,16
5,NumberOfTime60-89DaysPastDueNotWorse,0,0.000000,13
6,NumberOfTimes90DaysLate,0,0.000000,19
7,NumberRealEstateLoansOrLines,0,0.000000,28
8,RevolvingUtilizationOfUnsecuredLines,0,0.000000,125728
9,SeriousDlqin2yrs,0,0.000000,2


,warning,accepted_records
0,MonthlyIncome:MISSING_VALUE,29731
1,NumberOfDependents:MISSING_VALUE,3924
2,NumberOfTime30-59DaysPastDueNotWorse:UNUSUAL_V...,269
3,NumberOfTime60-89DaysPastDueNotWorse:UNUSUAL_V...,269
4,NumberOfTimes90DaysLate:UNUSUAL_VALUE,269
5,RevolvingUtilizationOfUnsecuredLines:UNUSUAL_V...,3321


,source_file,source_row,reason_codes,errors
0,cs-training.csv,65696,[OUT_OF_RANGE],"[{'code': 'OUT_OF_RANGE', 'field': 'age'}]"


Accepted target counts: {'SeriousDlqin2yrs': {'0': 139973, '1': 10026}}
Model features: ['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']


<div dir="rtl" style="direction: rtl; text-align: right; line-height: 1.9;">

## ۲. ساخت جدول داده‌های پذیرفته‌شده

<p dir="rtl" style="direction: rtl; text-align: right;">در این بخش، جدول <code dir="ltr">credit.accepted</code> در همان پایگاه داده پروژه ساخته می‌شود. تعریف جدول در فایل <code dir="ltr">sql/ddl/005_credit_accepted.sql</code> قرار دارد و دقیقاً شامل ۱۲ ستون داده‌های پذیرفته‌شده است. شناسه رکورد، کلید اصلی جدول است؛ ستون‌های درآمد و تعداد افراد تحت تکفل می‌توانند مقدار گمشده داشته باشند.</p>

<p dir="rtl" style="direction: rtl; text-align: right;">اتصال با تنظیمات موجود در فایل <code dir="ltr">.env</code> و تابع اتصال مشترک پروژه ساخته می‌شود. دستورهای ساخت جدول از طریق یک اتصال <bdi dir="ltr">SQLAlchemy</bdi> اجرا می‌شوند.</p>

</div>

In [73]:
from sqlalchemy import text
from IPython.display import Code, display
from data_factory.database.connection import create_database_engine

if "engine" in globals():
    engine.dispose()
engine = create_database_engine()

credit_ddl_path = PROJECT_ROOT / "sql/ddl/005_credit_accepted.sql"
credit_ddl = credit_ddl_path.read_text(encoding="utf-8")
display(Code(credit_ddl, language="sql"))

with engine.begin() as connection:
    connection.exec_driver_sql(
        credit_ddl, execution_options={"no_parameters": True}
    ).close()

print("Table ready: credit.accepted")

-- Stage 03: the twelve data fields in the validated accepted.csv.
-- Preserve nullable income/dependents and accepted unusual numeric values.

CREATE SCHEMA IF NOT EXISTS credit;

CREATE TABLE IF NOT EXISTS credit.accepted (
    row_id                                     BIGINT PRIMARY KEY CHECK (row_id > 0),
    "SeriousDlqin2yrs"                          BIGINT NOT NULL CHECK ("SeriousDlqin2yrs" IN (0, 1)),
    "RevolvingUtilizationOfUnsecuredLines"      DOUBLE PRECISION NOT NULL CHECK ("RevolvingUtilizationOfUnsecuredLines" >= 0),
    age                                        BIGINT NOT NULL CHECK (age > 0),
    "NumberOfTime30-59DaysPastDueNotWorse"       BIGINT NOT NULL CHECK ("NumberOfTime30-59DaysPastDueNotWorse" >= 0),
    "DebtRatio"                                DOUBLE PRECISION NOT NULL CHECK ("DebtRatio" >= 0),
    "MonthlyIncome"                            DOUBLE PRECISION CHECK ("MonthlyIncome" >= 0),
    "NumberOfOpenCreditLinesAndLoans"           BIGINT NOT NULL CHECK ("NumberOfOpenCreditLinesAndLoans" >= 0),
    "NumberOfTimes90DaysLate"                   BIGINT NOT NULL CHECK ("NumberOfTimes90DaysLate" >= 0),
    "NumberRealEstateLoansOrLines"              BIGINT NOT NULL CHECK ("NumberRealEstateLoansOrLines" >= 0),
    "NumberOfTime60-89DaysPastDueNotWorse"       BIGINT NOT NULL CHECK ("NumberOfTime60-89DaysPastDueNotWorse" >= 0),
    "NumberOfDependents"                       BIGINT CHECK ("NumberOfDependents" >= 0)
);

Table ready: credit.accepted


<div dir="rtl" style="direction: rtl; text-align: right; line-height: 1.9;">

## ۳. بارگذاری محتوای فایل پذیرفته‌شده با اتصال <bdi dir="ltr">SQLAlchemy</bdi>

<p dir="rtl" style="direction: rtl; text-align: right;">فایل <code dir="ltr">accepted.csv</code> از آخرین اجرای موفق لایه <bdi dir="ltr">Data Factory</bdi> خوانده می‌شود. سپس محتوای آن با <code dir="ltr">to_sql</code> و همان اتصال دیتابیس در جدول از پیش ساخته‌شده درج می‌شود. گزینه <code dir="ltr">index=False</code> از ایجاد ستون اضافه جلوگیری می‌کند.</p>

<p dir="rtl" style="direction: rtl; text-align: right;">مقادیر گمشده به‌صورت <code dir="ltr">NULL</code> ذخیره می‌شوند. نام ستون‌ها و مقادیر پذیرفته‌شده حفظ می‌شوند و جایگزینی مقدار گمشده یا مقیاس‌بندی انجام نمی‌شود. کل بارگذاری در یک تراکنش قرار دارد؛ در صورت خطا، درج رکوردهای همان اجرا برگشت داده می‌شود.</p>

</div>

In [74]:
credit_latest = json.loads((OUTPUT_ROOT / "latest.json").read_text(encoding="utf-8"))
credit_report_path = OUTPUT_ROOT / credit_latest["report"]
credit_report = json.loads(credit_report_path.read_text(encoding="utf-8"))
if credit_report["status"] != "passed":
    raise RuntimeError("The latest validation run did not pass; no data was loaded.")

accepted_csv_path = credit_report_path.parent / "accepted.csv"
accepted_contract = json.loads(
    (credit_report_path.parent / "contract.json").read_text(encoding="utf-8")
)
integer_columns = {
    field["name"]: "Int64"
    for field in accepted_contract["fields"] if field["type"] == "integer"
}
accepted_data = pd.read_csv(accepted_csv_path, dtype=integer_columns, float_precision="round_trip")

with engine.begin() as connection:
    accepted_data.to_sql(
        "accepted", con=connection, schema="credit",
        if_exists="append", index=False, chunksize=5000,
    )
    database_rows = connection.scalar(text("SELECT COUNT(*) FROM credit.accepted"))

print("Source:", accepted_csv_path.relative_to(PROJECT_ROOT))
print(f"Loaded: {len(accepted_data):,} rows into credit.accepted")
print(f"Table rows: {database_rows:,}")

Source: data\stage_03\curated\validation\daa6d8270f11093efe1e54c1614c3a3e348fffc2a4f876f533ad377289265259\accepted.csv
Loaded: 149,999 rows into credit.accepted
Table rows: 149,999


<div dir="rtl" style="direction: rtl; text-align: right; line-height: 1.9;">
 مرحله خواندن مستقیم دیتا از دیتابیس 

</div>

In [75]:
import pandas as pd
from sqlalchemy import text

with engine.connect() as connection:
    df = pd.read_sql_query(
        text("SELECT * FROM credit.accepted ORDER BY row_id"),
        con=connection,
        dtype_backend="numpy_nullable",
    )

print("Rows, columns:", df.shape)
display(df.head())

Rows, columns: (149999, 12)


,row_id,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1
2,3,0,0.65818,38,1,0.085113,3042.0,2,1,0,0,0
3,4,0,0.23381,30,0,0.03605,3300.0,5,0,0,0,0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0
